# Training on Kaggle (Tesla T4)

Trained from the GitHub repo — data and tokenizer are committed, so this
notebook only runs training and evaluation. Three runs:

1. + epochs 15, dropout 0.3, no label smoothing → overfit at epoch 3, BLEU 4.16
2. + dropout 0.4, label smoothing 0.1, patience 6 → best at epoch 15, BLEU 5.24
3. + epochs 30 → converged at epoch 29, BLEU 5.36 (final model)

Each change was diagnosed from the validation loss curve, not tuned blindly.

In [ ]:
!git clone https://github.com/MeMorphLing/Urdu-qgen-.git
%cd Urdu-qgen-
!pip install -q sentencepiece sacrebleu rouge-score
!python -c "import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))"

In [ ]:
!python -c "import torch; x=torch.randn(1000,1000,device='cuda'); print((x@x).sum().item())"
!python -m src.train --limit 10000 --epochs 1

In [ ]:
!python -m src.train --config configs/base.yaml

In [ ]:
!python -m src.evaluate --config configs/base.yaml

In [ ]:
!zip -qr run1_baseline.zip results checkpoints && ls -lh run1_baseline.zip

In [ ]:
!zip -qr run1_results_only.zip results && ls -lh run1_results_only.zip

In [ ]:
!git pull
!grep -E "dropout|label_smoothing|early_stop_patience" configs/base.yaml

In [ ]:
!python -m src.train --config configs/base.yaml

In [ ]:
!python -m src.evaluate --config configs/base.yaml

In [ ]:
!zip -qr run2_results.zip results && ls -lh run2_results.zip

In [ ]:
!zip -qr run2_full.zip results checkpoints

In [ ]:
!git pull
!grep "epochs:" configs/base.yaml

**Restarting and clear cell outputs**

In [1]:
%cd /kaggle/working/Urdu-qgen-
!pwd
!ls checkpoints results

/kaggle/working/Urdu-qgen-
/kaggle/working/Urdu-qgen-
checkpoints:
best.pt

results:
dataset_stats.json  metrics.json  tables.md
figures		    samples.tsv   train_log.csv


In [2]:
!python -m src.train --config configs/base.yaml

train pairs: 74379   valid pairs: 8212
trainable parameters: 31,173,440   ->  Table 2
/kaggle/working/Urdu-qgen-/src/train.py:104: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = (torch.cuda.amp.GradScaler()
epoch 01 | train 5.0470 (ppl 155.6) | valid 4.7067 (ppl 110.7) | lr 1.00e-03 | 160s
  saved best checkpoint -> checkpoints/best.pt
epoch 02 | train 4.3234 (ppl 75.4) | valid 4.5202 (ppl 91.9) | lr 1.00e-03 | 161s
  saved best checkpoint -> checkpoints/best.pt
epoch 03 | train 4.0528 (ppl 57.6) | valid 4.4737 (ppl 87.7) | lr 1.00e-03 | 161s
  saved best checkpoint -> checkpoints/best.pt
epoch 04 | train 3.9015 (ppl 49.5) | valid 4.4688 (ppl 87.3) | lr 1.00e-03 | 162s
  saved best checkpoint -> checkpoints/best.pt
epoch 05 | train 3.8056 (ppl 45.0) | valid 4.4703 (ppl 87.4) | lr 1.00e-03 | 162s
epoch 06 | train 3.7378 (ppl 42.0) | valid 4.4835 (ppl 88.5) | lr 1.00e-03 | 161s
epoch 07 | train 3.6

In [3]:
!python -m src.evaluate --config configs/base.yaml

loaded checkpoints/best.pt (epoch 29, val loss 4.3570)
uqa_valid: greedy done in 17s
uqa_valid: beam(5) x2000 in 50s
wiki_uqa: greedy done in 0s
wiki_uqa: beam(5) x178 in 5s

=== Table 3 ===
split       decode    BLEU-4  ROUGE-L     PPL    unk%
uqa_valid   greedy      5.35   0.2451   29.86   0.000
uqa_valid   beam        5.36   0.2542   29.86   0.000
wiki_uqa    greedy      3.53   0.2197   39.71   0.000
wiki_uqa    beam        4.09   0.2312   39.71   0.000

wrote results/metrics.json and results/samples.tsv


In [4]:
!python -m src.analyze
!python -m src.human_eval --make
!python -m src.viz --what loss
!zip -qr run3_results.zip results
!zip -qr run3_full.zip results checkpoints
!ls -lh run3_*.zip


=== uqa_valid ===
{
  "n": 2000,
  "question_words": {
    "kis (which/whom)": {
      "n": 650,
      "qword_accuracy": 43.8,
      "mean_sentBLEU": 7.08
    },
    "kya (what)": {
      "n": 586,
      "qword_accuracy": 53.2,
      "mean_sentBLEU": 6.55
    },
    "kaun (who)": {
      "n": 316,
      "qword_accuracy": 31.0,
      "mean_sentBLEU": 8.28
    },
    "kab (when)": {
      "n": 123,
      "qword_accuracy": 41.5,
      "mean_sentBLEU": 9.36
    },
    "kitne (how many)": {
      "n": 88,
      "qword_accuracy": 67.0,
      "mean_sentBLEU": 10.58
    },
    "kahan (where)": {
      "n": 56,
      "qword_accuracy": 23.2,
      "mean_sentBLEU": 8.94
    },
    "other/none": {
      "n": 47,
      "qword_accuracy": 4.3,
      "mean_sentBLEU": 4.64
    },
    "kitna (how much)": {
      "n": 47,
      "qword_accuracy": 61.7,
      "mean_sentBLEU": 12.41
    },
    "kitni (how much)": {
      "n": 37,
      "qword_accuracy": 21.6,
      "mean_sentBLEU": 10.95
    },
    "kyun (

In [5]:
import torch
ck = torch.load("checkpoints/best.pt", map_location="cpu", weights_only=False)
ck["optimizer"] = None
torch.save(ck, "best_inference.pt")
!ls -lh best_inference.pt

-rw-r--r-- 1 root root 119M Sep 12 09:51 best_inference.pt


In [7]:
!pwd
!ls -lh best_inference.pt
!ls -lh /kaggle/working/Urdu-qgen-/best_inference.pt

/kaggle/working/Urdu-qgen-
-rw-r--r-- 1 root root 119M Sep 12 09:51 best_inference.pt
-rw-r--r-- 1 root root 119M Sep 12 09:51 /kaggle/working/Urdu-qgen-/best_inference.pt


In [10]:
!cp best_inference.pt /kaggle/working/best_inference.pt
from IPython.display import FileLink
FileLink('/kaggle/working/best_inference.pt')

/kaggle/working/best_inference.pt

In [11]:
!split -b 40M /kaggle/working/best_inference.pt /kaggle/working/ckpt_part_
!ls -lh /kaggle/working/ckpt_part_*

-rw-r--r-- 1 root root 40M Sep 12 09:59 /kaggle/working/ckpt_part_aa
-rw-r--r-- 1 root root 40M Sep 12 09:59 /kaggle/working/ckpt_part_ab
-rw-r--r-- 1 root root 39M Sep 12 09:59 /kaggle/working/ckpt_part_ac
